In [1]:
import pandas as pd
import yfinance as yf
import requests
import warnings
warnings.filterwarnings("ignore")

print("OK")

OK


In [2]:
COMPANIES = {
    "SAP":       "SAP",
    "Oracle":    "ORCL",
    "Salesforce":"CRM",
    "Microsoft": "MSFT"
}

YEAR_START = "2019-01-01"
YEAR_END   = "2024-01-01"

print(f"Empresas: {list(COMPANIES.keys())}")
print(f"Período: {YEAR_START} → {YEAR_END}")

Empresas: ['SAP', 'Oracle', 'Salesforce', 'Microsoft']
Período: 2019-01-01 → 2024-01-01


In [4]:
prices_list = []

for name, ticker in COMPANIES.items():
    print(f"Descargando precios: {name} ({ticker})...")
    df = yf.download(ticker, start=YEAR_START, end=YEAR_END, auto_adjust=True, progress=False)
    
    # Aplanar columnas multi-nivel si existen
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    temp = df[["Close"]].copy()
    temp["company"] = name
    temp["ticker"]  = ticker
    temp = temp.reset_index()
    temp.columns = ["date", "close", "company", "ticker"]
    prices_list.append(temp)

prices = pd.concat(prices_list, ignore_index=True)

print(f"\nPrecios descargados: {len(prices)} filas")
print(prices.groupby("company")["close"].count())

Descargando precios: SAP (SAP)...
Descargando precios: Oracle (ORCL)...
Descargando precios: Salesforce (CRM)...
Descargando precios: Microsoft (MSFT)...

Precios descargados: 5032 filas
company
Microsoft     1258
Oracle        1258
SAP           1258
Salesforce    1258
Name: close, dtype: int64


In [5]:
financials = {}

for name, ticker in COMPANIES.items():
    print(f"Descargando financials: {name}...")
    stock = yf.Ticker(ticker)
    
    info = stock.info
    
    financials[name] = {
        "market_cap":        info.get("marketCap", None),
        "enterprise_value":  info.get("enterpriseValue", None),
        "revenue_ttm":       info.get("totalRevenue", None),
        "ebitda":            info.get("ebitda", None),
        "ev_revenue":        info.get("enterpriseToRevenue", None),
        "ev_ebitda":         info.get("enterpriseToEbitda", None),
        "pe_ratio":          info.get("trailingPE", None),
        "revenue_growth":    info.get("revenueGrowth", None),
        "gross_margins":     info.get("grossMargins", None),
    }

fin_df = pd.DataFrame(financials).T.reset_index()
fin_df.columns = ["company"] + list(fin_df.columns[1:])

print("\nMétricas financieras:")
print(fin_df[["company", "ev_revenue", "ev_ebitda", "pe_ratio"]].to_string(index=False))

Descargando financials: SAP...
Descargando financials: Oracle...
Descargando financials: Salesforce...
Descargando financials: Microsoft...

Métricas financieras:
   company  ev_revenue  ev_ebitda  pe_ratio
       SAP       5.019     16.151 22.867790
    Oracle       8.092     17.875 24.018835
Salesforce       3.894     12.932 19.225695
 Microsoft       9.262     15.982 23.257294


In [6]:
prices.to_csv("../data/prices.csv", index=False)
fin_df.to_csv("../data/financials.csv", index=False)

print("Guardado OK")
print(f"prices.csv: {len(prices)} filas")
print(f"financials.csv: {len(fin_df)} filas")

Guardado OK
prices.csv: 5032 filas
financials.csv: 4 filas
